In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [2]:
df = pd.read_csv('suicideRates.csv')

In [3]:
df.head()

,country,year,sex,age,suicides_no,population,suicides/100k pop,country-year,HDI for year,gdp_for_year ($),gdp_per_capita ($),generation
0,Albania,1987,male,15-24 years,21,312900,6.71,Albania1987,NaN,"2,156,624,900",796,Generation X
1,Albania,1987,male,35-54 years,16,308000,5.19,Albania1987,NaN,"2,156,624,900",796,Silent
2,Albania,1987,female,15-24 years,14,289700,4.83,Albania1987,NaN,"2,156,624,900",796,Generation X
3,Albania,1987,male,75+ years,1,21800,4.59,Albania1987,NaN,"2,156,624,900",796,G.I. Generation
4,Albania,1987,male,25-34 years,9,274300,3.28,Albania1987,NaN,"2,156,624,900",796,Boomers


One Hot Encoded Analysis

In [4]:
test_df = pd.DataFrame(data=[['male','15-24 years', 'Generation X']], columns=['sex', 'age','generation'])

In [5]:
def encode_onehot(df, test_data, column):
    target_column = 'suicides/100k pop'
    target_data = None
    if target_column in df.columns:
        target_data = df[target_column]
        df = df.drop(columns=[target_column])
    df_encoded = pd.get_dummies(df, columns=[column], prefix=column, prefix_sep=' - ', dtype=int)
    if target_data is not None:
        df_encoded[target_column] = target_data
    ohe_columns = [col for col in df_encoded.columns if col.startswith(column + " - ")]

    test_encoded = pd.get_dummies(test_data, columns=[column], prefix=column, prefix_sep=' - ', dtype=int)
    for col in ohe_columns:
        if col not in test_encoded:
            test_encoded[col] = 0

    test_encoded = test_encoded.reindex(columns=[col for col in df_encoded.columns if col != target_column], fill_value=0)

    return df_encoded, test_encoded

In [6]:
df_o = df.copy()
test_o = test_df.copy()
df_o = df_o.drop(['suicides_no', 'year','gdp_per_capita ($)', 'country-year', 'country', ' gdp_for_year ($) ', 'HDI for year', 'population'], axis=1)
df_o,test_o = encode_onehot(df_o,test_o, 'sex')
df_o,test_o = encode_onehot(df_o, test_o,'age')
df_o,test_o= encode_onehot(df_o, test_o,'generation')

In [7]:
df_o.head()

,sex - female,sex - male,age - 15-24 years,age - 25-34 years,age - 35-54 years,age - 5-14 years,age - 55-74 years,age - 75+ years,generation - Boomers,generation - G.I. Generation,generation - Generation X,generation - Generation Z,generation - Millenials,generation - Silent,suicides/100k pop
0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,6.71
1,0,1,0,0,1,0,0,0,0,0,0,0,0,1,5.19
2,1,0,1,0,0,0,0,0,0,0,1,0,0,0,4.83
3,0,1,0,0,0,0,0,1,0,1,0,0,0,0,4.59
4,0,1,0,1,0,0,0,0,1,0,0,0,0,0,3.28


In [8]:
test_o

,sex - female,sex - male,age - 15-24 years,age - 25-34 years,age - 35-54 years,age - 5-14 years,age - 55-74 years,age - 75+ years,generation - Boomers,generation - G.I. Generation,generation - Generation X,generation - Generation Z,generation - Millenials,generation - Silent
0,0,1,1,0,0,0,0,0,0,0,1,0,0,0


In [9]:
X = df_o.loc[:, df_o.columns != 'suicides/100k pop'].values
y = df_o.loc[:, df_o.columns == 'suicides/100k pop'].values.ravel()

In [10]:
from sklearn.linear_model import LinearRegression

In [11]:
LRmodel = LinearRegression()


In [12]:
LRmodel.fit(X,y)

LinearRegression()

In [13]:
y_pred = LRmodel.predict(test_o)

/Users/sohumpohane/.pyenv/versions/3.10.14/lib/python3.10/site-packages/sklearn/base.py:458: UserWarning: X has feature names, but LinearRegression was fitted without feature names
  warnings.warn(


In [14]:
print(f"Predicted Suicide/100k pop rate: {y_pred[0]}\n")

Predicted Suicide/100k pop rate: 17.04296875



In [15]:
def mae(_y, _y_pred):
    return (len(_y)**-1) * np.sum(np.abs(_y_pred-_y))

In [16]:
print(f"MAE: {mae(y, y_pred)}")

MAE: 14.45834871495327


In [17]:
print(f"Number of Regression Coefficients: {len(LRmodel.coef_)}")

Number of Regression Coefficients: 14


Numerical Features Analysis

In [18]:
from sklearn.preprocessing import LabelEncoder
def encode_categorical_features(train_df, test_df, additional_classes={}):
    for feature in train_df.columns.difference(['suicides/100k pop']):
        encoder = LabelEncoder()
        encoder.fit(train_df[feature])
        if feature in additional_classes:
            unique_classes = encoder.classes_.tolist() 
            unique_classes.extend(additional_classes[feature])
            encoder.classes_ = np.array(unique_classes)

        train_df[feature] = encoder.transform(train_df[feature])
        test_df[feature] = encoder.transform(test_df[feature])

    return train_df, test_df

In [19]:
df_o2 = df.copy()
test_o2 = test_df.copy()
df_o2 = df_o2.drop(['suicides_no', 'year','gdp_per_capita ($)', 'country-year', 'country', ' gdp_for_year ($) ', 'HDI for year', 'population'], axis=1)

In [20]:
df_o2

,sex,age,suicides/100k pop,generation
0,male,15-24 years,6.71,Generation X
1,male,35-54 years,5.19,Silent
2,female,15-24 years,4.83,Generation X
3,male,75+ years,4.59,G.I. Generation
4,male,25-34 years,3.28,Boomers
...,...,...,...,...
27815,female,35-54 years,2.96,Generation X
27816,female,75+ years,2.58,Silent
27817,male,5-14 years,2.17,Generation Z
27818,female,5-14 years,1.67,Generation Z


In [21]:
test_o2

,sex,age,generation
0,male,15-24 years,Generation X


In [22]:
df_o2, test_o2 = encode_categorical_features(df_o2, test_o2)


In [23]:
df_o2

,sex,age,suicides/100k pop,generation
0,1,0,6.71,2
1,1,2,5.19,5
2,0,0,4.83,2
3,1,5,4.59,1
4,1,1,3.28,0
...,...,...,...,...
27815,0,2,2.96,2
27816,0,5,2.58,5
27817,1,3,2.17,3
27818,0,3,1.67,3


In [24]:
test_o2

,sex,age,generation
0,1,0,2


In [25]:
X = df_o2.loc[:, df_o2.columns != 'suicides/100k pop'].values
y = df_o2.loc[:, df_o2.columns == 'suicides/100k pop'].values.ravel()

In [26]:
LRmodel2 = LinearRegression()

In [27]:
LRmodel2.fit(X,y)

LinearRegression()

In [28]:
y_pred = LRmodel2.predict(test_o2)

/Users/sohumpohane/.pyenv/versions/3.10.14/lib/python3.10/site-packages/sklearn/base.py:458: UserWarning: X has feature names, but LinearRegression was fitted without feature names
  warnings.warn(


In [29]:
print(f"Predicted Suicide/100k pop rate: {y_pred[0]}\n")

Predicted Suicide/100k pop rate: 15.178500202776474



In [30]:
print(f"MAE: {mae(y, y_pred)}")

MAE: 13.554067562029719


In [31]:
print(f"Number of Regression Coefficients: {len(LRmodel2.coef_)}")

Number of Regression Coefficients: 3


Changes between model performances:
The MAE for the model using one hot encoding was 14.458 while the MAE for the model using numerical features was 13.554. A lesser MAE means that the model's predictions aligned more with the actual data. Therefore the model using numerical features performed better. Also the number of regression coefficients for the better performing model was only 3 while the number of regression coefficients for the other model was 14

Using previous model for new prediction:

In [32]:
new_test_df = pd.DataFrame(data=[['male','25-34 years', 'Alpha']], columns=['sex', 'age','generation'])
additional_class = {'generation': ['Alpha']}

In [33]:
new_df = df.copy()
new_test_o = new_test_df.copy()
new_df = new_df.drop(['suicides_no', 'year','gdp_per_capita ($)', 'country-year', 'country', ' gdp_for_year ($) ', 'HDI for year', 'population'], axis=1)

In [34]:
new_df

,sex,age,suicides/100k pop,generation
0,male,15-24 years,6.71,Generation X
1,male,35-54 years,5.19,Silent
2,female,15-24 years,4.83,Generation X
3,male,75+ years,4.59,G.I. Generation
4,male,25-34 years,3.28,Boomers
...,...,...,...,...
27815,female,35-54 years,2.96,Generation X
27816,female,75+ years,2.58,Silent
27817,male,5-14 years,2.17,Generation Z
27818,female,5-14 years,1.67,Generation Z


In [35]:
new_test_o

,sex,age,generation
0,male,25-34 years,Alpha


In [36]:
new_df, new_test_o = encode_categorical_features(new_df, new_test_o, additional_classes = additional_class)

In [37]:
new_df

,sex,age,suicides/100k pop,generation
0,1,0,6.71,2
1,1,2,5.19,5
2,0,0,4.83,2
3,1,5,4.59,1
4,1,1,3.28,0
...,...,...,...,...
27815,0,2,2.96,2
27816,0,5,2.58,5
27817,1,3,2.17,3
27818,0,3,1.67,3


In [38]:
new_test_o

,sex,age,generation
0,1,1,6


In [39]:
X = new_df.loc[:, new_df.columns != 'suicides/100k pop'].values
y = new_df.loc[:, new_df.columns == 'suicides/100k pop'].values.ravel()

In [40]:
LRmodel3 = LinearRegression()

In [41]:
LRmodel3.fit(X,y)

LinearRegression()

In [42]:
y_pred = LRmodel3.predict(new_test_o)

/Users/sohumpohane/.pyenv/versions/3.10.14/lib/python3.10/site-packages/sklearn/base.py:458: UserWarning: X has feature names, but LinearRegression was fitted without feature names
  warnings.warn(


In [43]:
print(f"Predicted Suicide/100k pop rate: {y_pred[0]}\n")

Predicted Suicide/100k pop rate: 13.524885751088695



In [44]:
print(f"MAE: {mae(y, y_pred)}")

MAE: 12.847395212349253


Give one advantage when using regression (as opposed to classification with nominal features) in terms of independent variables

Regression allows you to find a continuous relationship between independed variables and dependent variables. When using classification, you are really only trying to put the dependent variable into categories which, doesn't allow you to truly get the continuous relationship if there is one.

Give one advantage when using regular numerical values rather than one-hot encoding for regression.

Numerical values preserve ordering of things, where as one hot encoding doesn't. When dealing with features such as age, you want to preserve this ordering such that you can capture the proper trend.

Now that you developed both a classifier (previously) and a regression model for the problem in this assignment, which method do you suggest to your machine learning model customer? Classifier or regression? Why?

I would suggest the regression model as we have the data on the exact rate which is a numerical value. Learning from this data and being able to predict on the rate is much more helpful and simply putting them in categories and predicting those categories. Making the suicide rate into categories makes you lose a lot of helpful data which a regression model could use to its advantage. I would also not use the one hot encoding, but rather converting variables into numerical values.